# Giai đoạn 2–3 — Chia Train/Test và đánh giá ban đầu ba mô hình

Theo kế hoạch dự án v3.1. Nhãn `recommended_career` mô tả **nhóm nghề nghiệp được khuyến nghị**, không phải tỷ lệ trúng tuyển đại học.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("Project:", ROOT)

## 1. X, y, LabelEncoder và chia đúng 80/20. Test được giữ riêng.

In [ ]:
import pandas as pd
from career_ml.data import load_data, prepare_xy, split_data
from career_ml.initial import run_initial_baselines
from career_ml.modeling import make_pipeline, feature_names_after_fs
X,y,encoder=prepare_xy(load_data())
X_train,X_test,y_train,y_test=split_data(X,y)
print('Train:',X_train.shape,'Test:',X_test.shape)
print('Labels:',encoder.classes_.tolist())
assert not set(X_train.index) & set(X_test.index)
print('Phân bố Train:'); print(y_train.value_counts().sort_index())

## 2. Kiểm tra pipeline (preprocess + selector + classifier) khi fit *chỉ Train*

In [ ]:
pipe=make_pipeline(X_train,"Logistic Regression","Filter")
pipe.fit(X_train,y_train)
print('Số đặc trưng được chọn:',len(feature_names_after_fs(pipe)))
print('Shape dự đoán:',pipe.predict(X_train.iloc[:3]).shape)
print('Shape xác suất:',pipe.predict_proba(X_train.iloc[:3]).shape)
# Không chạm tới Test trong cell này.

## 3. Đánh giá ba mô hình trước khi tinh chỉnh bằng 5-fold CV trên tập Train

In [ ]:
initial_results = run_initial_baselines(X_train, y_train, jobs=1)
columns = ["model", "fs", "cv_f1_macro", "cv_accuracy", "seconds"]
display(pd.DataFrame(initial_results)[columns].round(4))

Ba mô hình ở mục 3 dùng toàn bộ đặc trưng sau tiền xử lý. Notebook 04 so sánh 12 cấu hình gồm 3 mô hình × 4 trường hợp chọn đặc trưng.